In [ ]:
import os
import chromadb
from chromadb.config import Settings
import pandas as pd
from sentence_transformers import SentenceTransformer
import math

# ------------------ 1. เชื่อมต่อ ChromaDB ------------------
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# ------------------ 2. โหลด CSV ------------------
df = pd.read_csv("all_jobs.csv")  # ต้องมี columns: jobpost_id, detail, position, job_id, company_id
print(f"📊 โหลดข้อมูล {len(df)} รายการ")

# ------------------ 3. Clean data ------------------
df = df.dropna(subset=["detail"])               # ลบ row ที่ detail ว่าง
df = df.drop_duplicates(subset=["jobpost_id"])  # ลบ duplicate jobpost_id
print(f"🧹 ทำความสะอาดข้อมูลเหลือ {len(df)} รายการ")

# ------------------ 4. เตรียม embeddings ------------------
print("🤖 กำลังโหลดโมเดล...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("🔄 กำลังสร้าง embeddings...")
documents = df["detail"].tolist()
embeddings = model.encode(documents, show_progress_bar=True).tolist()

ids = df["jobpost_id"].astype(str).tolist()
metadatas = df.drop(columns=["jobpost_id", "detail"]).to_dict(orient="records")

# ------------------ 5. ตรวจสอบความยาว ------------------
assert len(ids) == len(documents) == len(embeddings) == len(metadatas), "Length mismatch!"
print(f"✅ เตรียมข้อมูลสำเร็จ: {len(ids)} รายการ")

# ------------------ 6. สร้าง collection หรือใช้ที่มีอยู่ ------------------
collection_name = "job_postings_vector"
if collection_name in [c.name for c in client.list_collections()]:
    collection = client.get_collection(collection_name)
    print(f"📦 ใช้ collection ที่มีอยู่: {collection_name}")
else:
    collection = client.create_collection(name=collection_name)
    print(f"📦 สร้าง collection ใหม่: {collection_name}")

# ------------------ 7. Upsert ข้อมูลแบบ batch ------------------
BATCH_SIZE = 5000  # ไม่เกิน max_batch_size ของ ChromaDB
n = len(ids)
num_batches = math.ceil(n / BATCH_SIZE)

print(f"\n🚀 กำลัง upsert {n} รายการ ({num_batches} batches)...")

for i in range(num_batches):
    start = i * BATCH_SIZE
    end = min((i + 1) * BATCH_SIZE, n)
    
    collection.upsert(
        ids=ids[start:end],
        documents=documents[start:end],
        embeddings=embeddings[start:end],
        metadatas=metadatas[start:end]
    )
    print(f"✅ Upsert batch {i+1}/{num_batches} สำเร็จ! ({start+1}-{end})")

print(f"\n🎉 Upsert ข้อมูลสำเร็จทั้งหมด!")

In [ ]:
# ดู collections
collections = client.list_collections()
print(f"Collections: {[c.name for c in collections]}")

In [ ]:
# ------------------ 8. ทดสอบ Query ------------------
queries = [
    # ค้นหาจากหน้าที่และความรับผิดชอบ
    "ออกแบบ database เขียน API backend",
    "ดูแลระบบ server network infrastructure",
    "ประสานงานลูกค้า นำเสนอขาย",
    "วิเคราะห์ข้อมูล สร้าง report dashboard",
    "จัดการ social media content marketing",
    
    # ค้นหาจากทักษะ
    "Python Django PostgreSQL REST API",
    "React TypeScript Next.js frontend",
    "digital marketing SEO Google Ads",
    "Excel PowerBI data analysis",
    "Photoshop Illustrator graphic design",
    
    # ค้นหาจากลักษณะงาน
    "work from home remote",
    "รับเด็กจบใหม่ ไม่ต้องมีประสบการณ์",
    "งานด่วน เริ่มงานทันที",
    "เงินเดือน 30000-50000",
    "ทำงาน 5 วัน ประกันสังคม",
]

print("\n🔍 กำลังสร้าง query embeddings...")
query_embs = model.encode(queries, show_progress_bar=True).tolist()

# ------------------ 9. แสดงผลลัพธ์ ------------------
for q, emb in zip(queries, query_embs):
    results = collection.query(
        query_embeddings=[emb],  # ต้องอยู่ใน list
        n_results=5              # จำนวนผลลัพธ์ที่ต้องการ
    )
    
    print(f"\n{'='*60}")
    print(f"🔎 Query: {q}")
    print(f"{'='*60}")
    
    # ตรวจสอบว่ามีผลลัพธ์หรือไม่
    if not results["ids"] or not results["ids"][0]:
        print("❌ ไม่พบผลลัพธ์")
        continue
    
    for i, doc_id in enumerate(results["ids"][0]):
        distance = results["distances"][0][i] if "distances" in results else None
        
        print(f"\n📌 Result {i+1}")
        print(f"   JobPost ID: {doc_id}")
        if distance is not None:
            print(f"   Distance: {distance:.4f}")
        print(f"   Detail: {results['documents'][0][i][:200]}...")  # แสดงแค่ 200 ตัวอักษรแรก
        print(f"   Metadata: {results['metadatas'][0][i]}")

print("\n✨ เสร็จสิ้นการทดสอบ!")